# Task 4.2 — Adversarial Noise Injection: FGSM

In [12]:
!pip install -q transformers soundfile torchaudio matplotlib numpy

In [13]:
import gc, json, math
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torchaudio
from transformers import AutoFeatureExtractor, Wav2Vec2Model

TARGET_SR = 16_000
ENGLISH, HINDI = 0, 1
IDX_TO_LANG = {ENGLISH: 'English', HINDI: 'Hindi'}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## LID Head (same architecture as Task 1.1)

In [14]:
class LIDHead(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, out_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)

def load_mono_16k(path):
    wav_np, sr = sf.read(str(path), always_2d=True, dtype='float32')
    wav = torch.from_numpy(wav_np.T)
    if wav.size(0) > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    return wav.squeeze(0).contiguous()

def compute_snr(original, perturbation):
    sig_p = torch.mean(original**2).item()
    noise_p = torch.mean(perturbation**2).item()
    if noise_p < 1e-20: return float('inf')
    return 10.0 * math.log10(sig_p / (noise_p + 1e-20))

## Load audio & models

In [ ]:
# Load audio
audio_path = 'original_segment.wav'
assert Path(audio_path).exists(), f'Please upload {audio_path}!'
full_wav = load_mono_16k(audio_path)

# Extract 5-second segment
start_s, duration_s = 30.0, 5.0
start_sample = int(start_s * TARGET_SR)
end_sample = min(start_sample + int(duration_s * TARGET_SR), full_wav.numel())
segment = full_wav[start_sample:end_sample]
print(f'Segment: {start_s}s to {end_sample/TARGET_SR:.1f}s ({segment.numel()/TARGET_SR:.1f}s)')

# Load wav2vec2
wav2vec_model_name = 'facebook/wav2vec2-base'
print(f'Loading {wav2vec_model_name}...')
feature_extractor = AutoFeatureExtractor.from_pretrained(wav2vec_model_name)
wav2vec = Wav2Vec2Model.from_pretrained(wav2vec_model_name).to(device)
wav2vec.eval()
for p in wav2vec.parameters():
    p.requires_grad = False

# Load LID head
lid_head = LIDHead().to(device)
lid_path = 'lid_model.pt'
if Path(lid_path).exists():
    lid_head.load_state_dict(torch.load(lid_path, map_location=device))
    print(f'Loaded LID head from {lid_path}')
else:
    print(f'Warning: {lid_path} not found, using random weights')
lid_head.eval()

Segment: 30.0s to 35.0s (5.0s)
Loading facebook/wav2vec2-base...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded LID head from lid_model.pt


LIDHead(
  (net): Sequential(
    (0): Linear(in_features=768, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=2, bias=True)
  )
)

## Get original prediction

In [16]:
with torch.no_grad():
    inputs = feature_extractor(segment.numpy(), sampling_rate=TARGET_SR, return_tensors='pt')
    out = wav2vec(input_values=inputs.input_values.to(device))
    features = out.last_hidden_state.squeeze(0)
    logits = lid_head(features)
    preds = torch.argmax(logits, dim=-1)
    hindi_pct = (preds == HINDI).float().mean().item() * 100
    english_pct = (preds == ENGLISH).float().mean().item() * 100

print(f'Original: Hindi={hindi_pct:.1f}%, English={english_pct:.1f}%')
source_label = HINDI if hindi_pct > english_pct else ENGLISH
target_label = ENGLISH if source_label == HINDI else HINDI
print(f'Attack: flip {IDX_TO_LANG[source_label]} → {IDX_TO_LANG[target_label]}')

Original: Hindi=100.0%, English=0.0%
Attack: flip Hindi → English


## FGSM Attack (GPU-accelerated)

In [17]:
def fgsm_attack(waveform, epsilon, target_lbl):
    """Apply FGSM to flip LID prediction."""
    # Create differentiable input
    inputs = feature_extractor(waveform.numpy(), sampling_rate=TARGET_SR, return_tensors='pt')
    input_values = inputs.input_values.to(device)
    input_values.requires_grad_(True)
    input_values.retain_grad()


    # Forward
    out = wav2vec(input_values=input_values)
    features = out.last_hidden_state.squeeze(0)
    logits = lid_head(features)
    target = torch.full((logits.size(0),), target_lbl, dtype=torch.long, device=device)
    loss = nn.CrossEntropyLoss()(logits, target)
    loss.backward()

    # Perturbation
    if input_values.grad is not None:
        perturbation = epsilon * input_values.grad.sign().squeeze(0).cpu()
    else:
        perturbation = torch.zeros_like(waveform)

    perturbed = waveform + perturbation[:waveform.numel()]

    # Check success
    with torch.no_grad():
        adv_inputs = feature_extractor(perturbed.numpy(), sampling_rate=TARGET_SR, return_tensors='pt')
        adv_out = wav2vec(input_values=adv_inputs.input_values.to(device))
        adv_logits = lid_head(adv_out.last_hidden_state.squeeze(0))
        adv_preds = torch.argmax(adv_logits, dim=-1)
        n_target = (adv_preds == target_lbl).sum().item()
        success = n_target > adv_preds.numel() * 0.5

    return perturbed, perturbation[:waveform.numel()], success

## Epsilon search

In [25]:
epsilon_range = [0.0001, 0.0003, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
results = []

for eps in epsilon_range:
    perturbed, perturbation, success = fgsm_attack(segment, eps, target_label)
    snr = compute_snr(segment, perturbation)
    results.append({
        'epsilon': eps, 'success': success,
        'snr_db': round(snr, 2), 'inaudible': snr > 40.0,
    })
    status = 'FLIPPED' if success else 'unchanged'
    audible = 'inaudible' if snr > 40 else 'AUDIBLE'
    print(f'  ε={eps:.4f} → {status}, SNR={snr:.1f}dB ({audible})')

  ε=0.0001 → unchanged, SNR=45.9dB (inaudible)
  ε=0.0003 → unchanged, SNR=36.4dB (AUDIBLE)
  ε=0.0005 → unchanged, SNR=31.9dB (AUDIBLE)
  ε=0.0010 → unchanged, SNR=25.9dB (AUDIBLE)
  ε=0.0020 → unchanged, SNR=19.9dB (AUDIBLE)
  ε=0.0050 → unchanged, SNR=11.9dB (AUDIBLE)
  ε=0.0100 → unchanged, SNR=5.9dB (AUDIBLE)
  ε=0.0200 → unchanged, SNR=-0.1dB (AUDIBLE)
  ε=0.0500 → unchanged, SNR=-8.1dB (AUDIBLE)


## Analysis & save results

In [26]:
successful = [r for r in results if r['success']]
inaud_success = [r for r in successful if r['inaudible']]

summary = {
    'segment_start_s': start_s, 'segment_duration_s': duration_s,
    'source_language': IDX_TO_LANG[source_label],
    'target_language': IDX_TO_LANG[target_label],
    'original_hindi_pct': round(hindi_pct, 1),
    'original_english_pct': round(english_pct, 1),
    'epsilon_results': results,
}

if inaud_success:
    min_eps = min(r['epsilon'] for r in inaud_success)
    summary['min_epsilon_inaudible'] = min_eps
    summary['min_epsilon_snr_db'] = next(r['snr_db'] for r in inaud_success if r['epsilon']==min_eps)
    print(f'✓ Minimum inaudible epsilon: {min_eps}, SNR: {summary["min_epsilon_snr_db"]} dB')
elif successful:
    min_eps = min(r['epsilon'] for r in successful)
    summary['min_epsilon'] = min_eps
    print(f'△ Flip at ε={min_eps} but SNR < 40dB')
else:
    print('✗ Could not flip in tested range')

with open('adversarial_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Save best perturbed audio
if successful:
    best_eps = min(r['epsilon'] for r in successful)
    perturbed, _, _ = fgsm_attack(segment, best_eps, target_label)
    sf.write('adversarial_segment.wav', perturbed.detach().numpy(), TARGET_SR)
    print('Saved adversarial_segment.wav')

✗ Could not flip in tested range


In [27]:
# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)

epsilons = [r['epsilon'] for r in results]
snrs = [r['snr_db'] for r in results]
successes = [r['success'] for r in results]

colors = ['green' if s else 'red' for s in successes]
axes[0].bar(range(len(epsilons)), snrs, color=colors, alpha=0.7)
axes[0].axhline(y=40, color='orange', linestyle='--', label='SNR=40dB')
axes[0].set_xticks(range(len(epsilons)))
axes[0].set_xticklabels([f'{e:.4f}' for e in epsilons], rotation=45)
axes[0].set_xlabel('Epsilon'); axes[0].set_ylabel('SNR (dB)')
axes[0].set_title('FGSM: SNR vs Epsilon (Green=flipped, Red=unchanged)')
axes[0].legend(); axes[0].grid(alpha=0.2)

axes[1].plot(epsilons, snrs, 'o-', linewidth=2)
axes[1].axhline(y=40, color='orange', linestyle='--', label='Inaudibility threshold')
axes[1].set_xlabel('Epsilon'); axes[1].set_ylabel('SNR (dB)')
axes[1].set_title('SNR vs Perturbation Strength');
axes[1].legend(); axes[1].grid(alpha=0.2)

fig.suptitle('Adversarial Robustness Analysis (FGSM)', fontsize=14)
fig.savefig('adversarial_analysis.png', dpi=200)
plt.show()
print('Done!')

Done!
